In [ ]:
import json
from tqdm import tqdm
import random


def flatten_table(table):
    return "\n".join(
        " | ".join(str(cell).strip() for cell in row)
        for row in table
    )


def build_dataset(samples):
    new_data = []
    id_sample = 0
    for sample in samples:

        annotation = sample["annotation"]

        # turn hiện tại
        turn_ind = annotation["turn_ind"]

        questions = annotation["dialogue_break"]
        answers = annotation["exe_ans_list"]
        print(list(sample.keys()))
        label = sample["qa"]["program"]
        
        pre_text = " ".join(sample["pre_text"])
        post_text = " ".join(sample["post_text"])
        table_text = flatten_table(sample["table"])

        context_parts = [
            pre_text,
            table_text,
            post_text,
        ]

        # lịch sử QA trước turn hiện tại
        for i in range(turn_ind):
            context_parts.append(
                f"{questions[i]} {answers[i]}"
            )

        # câu hỏi hiện tại
        context_parts.append(
            f"{questions[turn_ind]}"
        )
        new_data.append({
            "input": "\n\n".join(context_parts),
            "label": str(label)
        })

    return new_data




In [7]:
with open("dev_ori.json", "r", encoding="utf-8") as f:
    data = json.load(f)
sample = data[0]

In [10]:
sample["qa"]["program"]

'subtract(60.94, 25.14), divide(#0, 25.14)'

In [12]:
# Load data gốc
with open("dev_ori.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Build dataset mới
new_data = build_dataset(data[:10])
random.seed(42)      
random.shuffle(new_data)

for idx, item in enumerate(new_data):
    item["id"] = idx


KeyError: 'qa'

In [4]:
new_data[0]

{'input': 'as of december 31 , 2017 , the company had gross state income tax credit carry-forwards of approximately $ 20 million , which expire from 2018 through 2020 . a deferred tax asset of approximately $ 16 million ( net of federal benefit ) has been established related to these state income tax credit carry-forwards , with a valuation allowance of $ 7 million against such deferred tax asset as of december 31 , 2017 . the company had a gross state net operating loss carry-forward of $ 39 million , which expires in 2027 . a deferred tax asset of approximately $ 3 million ( net of federal benefit ) has been established for the net operating loss carry-forward , with a full valuation allowance as of december 31 , 2017 . other state and foreign net operating loss carry-forwards are separately and cumulatively immaterial to the company 2019s deferred tax balances and expire between 2026 and 2036 . 14 . debt long-term debt consisted of the following: .\n\n( $ in millions ) | december 31

In [ ]:
# Save
with open("new_data.json", "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(new_data)} samples to new_data.json")